# Movie Rating Analysis Using Python
**CODTECH Solutions — Data Science Internship Project**

## Project objective
Analyze movie ratings to discover highly rated movies, popular genres, rating distributions, and user rating behavior using Python.

## Dataset
Use the MovieLens Latest Small dataset from GroupLens Research:
https://grouplens.org/datasets/movielens/latest/

Download `ml-latest-small.zip`, extract it, and upload both `movies.csv` and `ratings.csv` when prompted. The dataset contains movie metadata and user ratings; dataset details can change, so this notebook reports the actual counts from the uploaded files.

## Tools
Python, Pandas, NumPy, Matplotlib, and Seaborn.

**How to run:** Run each cell from top to bottom in Google Colab. Upload the two CSV files when prompted.


In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import io

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")


Libraries imported successfully.


## 1. Upload the dataset files

In [ ]:
uploaded = files.upload()

# Read the uploaded CSV files by filename
csv_data = {}
for filename, content in uploaded.items():
    if filename.lower().endswith(".csv"):
        csv_data[filename.lower()] = pd.read_csv(io.BytesIO(content))

movies_key = next((k for k in csv_data if k.endswith("movies.csv")), None)
ratings_key = next((k for k in csv_data if k.endswith("ratings.csv")), None)

if movies_key is None or ratings_key is None:
    raise ValueError("Please upload both movies.csv and ratings.csv from the extracted MovieLens dataset.")

movies = csv_data[movies_key].copy()
ratings = csv_data[ratings_key].copy()

print("Movies dataset shape:", movies.shape)
print("Ratings dataset shape:", ratings.shape)
display(movies.head())
display(ratings.head())


## 2. Explore and validate the data

In [ ]:
print("Movie columns:", movies.columns.tolist())
print("Rating columns:", ratings.columns.tolist())

print("\nMissing values in movies:")
display(movies.isnull().sum().to_frame("Missing values"))

print("\nMissing values in ratings:")
display(ratings.isnull().sum().to_frame("Missing values"))

print("\nDuplicate movie rows:", movies.duplicated().sum())
print("Duplicate rating rows:", ratings.duplicated().sum())

movies = movies.drop_duplicates(subset=["movieId"]).copy()
ratings = ratings.drop_duplicates().copy()

ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")
ratings = ratings.dropna(subset=["movieId", "rating"])
print("Cleaned movie rows:", len(movies))
print("Cleaned rating rows:", len(ratings))


## 3. Combine movie titles with ratings

In [ ]:
movie_ratings = ratings.merge(movies, on="movieId", how="left")

print("Combined dataset shape:", movie_ratings.shape)
display(movie_ratings.head())
print("\nRating summary:")
display(movie_ratings["rating"].describe())


## 4. Overall rating distribution

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(data=ratings, x="rating", order=sorted(ratings["rating"].dropna().unique()))
plt.title("Distribution of Movie Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of ratings")
plt.tight_layout()
plt.show()


## 5. Top-rated movies (with a minimum rating count)

In [ ]:
# Requiring a minimum number of ratings reduces the effect of movies rated by only a few people.
MIN_RATINGS = 30

movie_summary = (
    movie_ratings.groupby(["movieId", "title"], dropna=False)
    .agg(average_rating=("rating", "mean"),
         rating_count=("rating", "count"))
    .reset_index()
)

eligible_movies = movie_summary[movie_summary["rating_count"] >= MIN_RATINGS]
top_rated = eligible_movies.sort_values(
    ["average_rating", "rating_count"], ascending=[False, False]
).head(15)

print(f"Top movies with at least {MIN_RATINGS} ratings:")
display(top_rated[["title", "average_rating", "rating_count"]].round(2))

if not top_rated.empty:
    plot_data = top_rated.sort_values("average_rating")
    plt.figure(figsize=(11, 7))
    sns.barplot(data=plot_data, x="average_rating", y="title")
    plt.title(f"Top-Rated Movies (At Least {MIN_RATINGS} Ratings)")
    plt.xlabel("Average rating")
    plt.ylabel("Movie")
    plt.xlim(0, 5)
    plt.tight_layout()
    plt.show()


## 6. Most-rated movies

In [ ]:
most_rated = movie_summary.sort_values(
    ["rating_count", "average_rating"], ascending=[False, False]
).head(15)

display(most_rated[["title", "rating_count", "average_rating"]].round(2))

plt.figure(figsize=(11, 7))
sns.barplot(data=most_rated.sort_values("rating_count"), x="rating_count", y="title")
plt.title("Most-Rated Movies")
plt.xlabel("Number of ratings")
plt.ylabel("Movie")
plt.tight_layout()
plt.show()


## 7. Genre analysis

In [ ]:
# The MovieLens genres column uses pipe-separated labels; '(no genres listed)' is excluded.
genre_ratings = movie_ratings[["rating", "genres"]].dropna().copy()
genre_ratings["genre"] = genre_ratings["genres"].str.split("|")
genre_ratings = genre_ratings.explode("genre")
genre_ratings = genre_ratings[genre_ratings["genre"] != "(no genres listed)"]

genre_summary = (
    genre_ratings.groupby("genre")
    .agg(average_rating=("rating", "mean"),
         rating_count=("rating", "count"))
    .sort_values("average_rating", ascending=False)
)

print("Genre summary (sorted by average rating):")
display(genre_summary.round(2))

plt.figure(figsize=(10, 7))
sns.barplot(x=genre_summary["average_rating"].values, y=genre_summary.index)
plt.title("Average Rating by Genre")
plt.xlabel("Average rating")
plt.ylabel("Genre")
plt.xlim(0, 5)
plt.tight_layout()
plt.show()


## 8. Ratings by year (where movie year is available in title)

In [ ]:
# Extract a four-digit year at the end of movie titles when present.
movie_years = movies[["movieId", "title"]].copy()
movie_years["year"] = movie_years["title"].str.extract(r"\((\d{4})\)\s*$")
movie_years["year"] = pd.to_numeric(movie_years["year"], errors="coerce")

ratings_by_year = ratings.merge(movie_years[["movieId", "year"]], on="movieId", how="left")
year_summary = ratings_by_year.dropna(subset=["year"]).groupby("year").agg(
    average_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()

# Plot only years with at least 20 ratings to reduce noise.
year_plot = year_summary[year_summary["rating_count"] >= 20].sort_values("year")
display(year_plot.tail(15).round(2))

plt.figure(figsize=(12, 5))
sns.lineplot(data=year_plot, x="year", y="average_rating")
plt.title("Average Movie Rating by Release Year")
plt.xlabel("Release year")
plt.ylabel("Average rating")
plt.tight_layout()
plt.show()


## 9. User rating activity

In [ ]:
user_activity = ratings.groupby("userId").agg(
    ratings_given=("rating", "count"),
    average_rating=("rating", "mean")
).sort_values("ratings_given", ascending=False)

print("Most active users by number of ratings:")
display(user_activity.head(10).round(2))

plt.figure(figsize=(9, 5))
sns.histplot(user_activity["ratings_given"], bins=30, kde=False)
plt.title("Distribution of Ratings per User")
plt.xlabel("Number of ratings given by a user")
plt.ylabel("Number of users")
plt.tight_layout()
plt.show()


## 10. Automated findings summary

In [ ]:
print("=" * 55)
print("MOVIE RATING ANALYSIS — SUMMARY")
print("=" * 55)
print(f"Movies in dataset: {movies['movieId'].nunique():,}")
print(f"Ratings analyzed: {len(ratings):,}")
print(f"Unique users: {ratings['userId'].nunique():,}")
print(f"Average rating across all ratings: {ratings['rating'].mean():.2f} / 5")
print(f"Median rating: {ratings['rating'].median():.1f} / 5")

if not movie_summary.empty:
    most_rated_row = movie_summary.sort_values("rating_count", ascending=False).iloc[0]
    print(f"Most-rated movie: {most_rated_row['title']} ({int(most_rated_row['rating_count'])} ratings)")

if not eligible_movies.empty:
    best_row = eligible_movies.sort_values(
        ["average_rating", "rating_count"], ascending=[False, False]
    ).iloc[0]
    print(f"Highest-rated movie with at least {MIN_RATINGS} ratings: {best_row['title']} "
          f"({best_row['average_rating']:.2f}/5, {int(best_row['rating_count'])} ratings)")

if not genre_summary.empty:
    print(f"Highest average-rated genre in this dataset: {genre_summary.index[0]} "
          f"({genre_summary.iloc[0]['average_rating']:.2f}/5)")

print("=" * 55)


## 11. Export summary files (optional)


In [ ]:
movie_summary.sort_values("average_rating", ascending=False).to_csv("movie_rating_summary.csv", index=False)
genre_summary.to_csv("genre_rating_summary.csv")
print("Exported movie_rating_summary.csv and genre_rating_summary.csv")

# Uncomment to download a file to your computer:
# files.download("movie_rating_summary.csv")
# files.download("genre_rating_summary.csv")


## 12. Conclusion

This project demonstrates exploratory data analysis of movie ratings. It combines movie metadata with user ratings, checks data quality, examines rating distributions, ranks movies while applying a minimum-rating threshold, compares genres, and studies rating activity.

Use the actual outputs from your run to describe your findings. Ratings reflect the users represented in MovieLens and should not be treated as universal audience opinion.

## References
- MovieLens datasets from GroupLens Research: https://grouplens.org/datasets/movielens/
- Pandas documentation: https://pandas.pydata.org/docs/
- Matplotlib documentation: https://matplotlib.org/stable/
- Seaborn documentation: https://seaborn.pydata.org/
